# 🗑️ Safe Firebase Variable Deletion Tool\n

## 1. Imports & Firebase Initialization\n
Initialize the Firebase connection securely.\n

In [1]:
import os
import json
import logging
from datetime import datetime
import pandas as pd
import firebase_admin
from firebase_admin import credentials, db
import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

# Create Logger
logging.basicConfig(
    filename='deletion_operations.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logging.info("Notebook started.")

# Initialize Firebase
cred_path = "D:/staklimjerukagung-firebase-adminsdk-kcfma-e091165a9b.json"
database_url = "https://staklimjerukagung-default-rtdb.asia-southeast1.firebasedatabase.app/"

try:
    firebase_admin.get_app()
    print("✅ Firebase already initialized.")
except ValueError:
    cred = credentials.Certificate(cred_path)
    firebase_admin.initialize_app(cred, {'databaseURL': database_url})
    print("✅ Firebase initialized successfully.")

✅ Firebase initialized successfully.


## 2. Configuration\n
Set the target variable, stations, and time range.\n

In [11]:
# ==========================================
# 🎯 TARGET CONFIGURATION
# ==========================================
TARGET_VARIABLE = "temp", "humi", "pres", # Example: "soil_temp", "soil_moisture", "battery_voltage"

# Define stations: ["id-01", "id-02"] or ["ALL"]
TARGET_STATIONS = ["id-03"]

# Define time range (YYYY-MM-DD). Set to None for all time.
START_DATE = "2023-01-01"
END_DATE = "2026-12-31"

BATCH_SIZE = 5000 # Number of updates per Firebase batch

# Safety Check
CRITICAL_VARS = ["rainfall", "air_temp", "humidity", "pressure"]
if TARGET_VARIABLE in CRITICAL_VARS:
    print(f"⚠️ WARNING: You are targeting a CRITICAL meteorological variable ({TARGET_VARIABLE}).")
    confirm = input("Are you sure? Type 'YES' to proceed: ")
    if confirm != 'YES':
        raise Exception("Operation aborted by user.")
print(f"Target: {TARGET_VARIABLE}")

def date_to_unix(date_str):
    if not date_str: return None
    import datetime as dt
    d = dt.datetime.strptime(date_str, "%Y-%m-%d")
    return str(int(d.timestamp()))

start_unix = date_to_unix(START_DATE)
end_unix = date_to_unix(END_DATE)

Target: ('temp', 'humi', 'pres')


## 3. Chunked Firebase Scanner\n
Fetch data safely without causing Out-Of-Memory (OOM) errors.\n

In [12]:
def get_target_stations():
    if "ALL" in TARGET_STATIONS:
        ref = db.reference('auto_weather_stat')
        return list(ref.get(shallow=True).keys())
    return TARGET_STATIONS

def fetch_affected_records(station_id, target_var, start_ts=None, end_ts=None, chunk_size=10000):
    ref = db.reference(f'auto_weather_stat/{station_id}/data')
    affected_records = {}
    
    query = ref.order_by_key().limit_to_first(chunk_size)
    if start_ts:
        query = ref.order_by_key().start_at(start_ts).limit_to_first(chunk_size)
        
    while True:
        try:
            snapshot = query.get()
        except Exception as e:
            logging.error(f"Error fetching data: {e}")
            break
            
        if not snapshot:
            break
            
        keys = list(snapshot.keys())
        for ts, data in snapshot.items():
            if end_ts and ts > end_ts:
                snapshot = {}
                break
                
            if isinstance(data, dict) and target_var in data:
                affected_records[ts] = data[target_var]
                
        if not snapshot:
            break
            
        last_key = keys[-1]
        if len(keys) < chunk_size:
            break
            
        query = ref.order_by_key().start_at(last_key).limit_to_first(chunk_size + 1)
        
    return affected_records

## 4. Dry Run Mode (Mandatory)\n
Scan the database and report the number of affected records without making any modifications.\n

In [13]:
print("🔍 INITIATING DRY RUN MODE...")
stations = get_target_stations()
total_affected = 0
all_affected_data = {}

for station in stations:
    print(f"Scanning {station}...")
    affected = fetch_affected_records(station, TARGET_VARIABLE, start_unix, end_unix)
    all_affected_data[station] = affected
    count = len(affected)
    total_affected += count
    print(f"  -> Found {count} records containing '{TARGET_VARIABLE}'")

print("="*40)
print(f"Dry Run Complete.")
print(f"Target Variable: {TARGET_VARIABLE}")
print(f"Affected Stations: {stations}")
print(f"Total timestamps affected: {total_affected}")
print("No data was deleted.")

🔍 INITIATING DRY RUN MODE...
Scanning id-03...
  -> Found 0 records containing '('temp', 'humi', 'pres')'
Dry Run Complete.
Target Variable: ('temp', 'humi', 'pres')
Affected Stations: ['id-03']
Total timestamps affected: 0
No data was deleted.


## 5. Backup Generation (Parquet)\n
Save affected records locally before deletion to enable rollback.\n

In [9]:
BACKUP_DIR = "backup_deletions"
os.makedirs(BACKUP_DIR, exist_ok=True)

backup_files = {}

for station, records in all_affected_data.items():
    if not records: continue
    
    df = pd.DataFrame(list(records.items()), columns=["timestamp", TARGET_VARIABLE])
    df["station"] = station
    df["deletion_date"] = datetime.now().isoformat()
    
    file_path = os.path.join(BACKUP_DIR, f"backup_{station}_{TARGET_VARIABLE}.parquet")
    df.to_parquet(file_path, index=False)
    backup_files[station] = file_path
    
    meta = {
        "station": station,
        "variable": TARGET_VARIABLE,
        "count": len(records),
        "backup_date": datetime.now().isoformat(),
        "parquet_file": file_path
    }
    with open(file_path.replace(".parquet", ".json"), "w") as f:
        json.dump(meta, f, indent=2)
        
    print(f"✅ Saved backup for {station} to {file_path}")

logging.info(f"Backup completed. Files: {backup_files}")

## 6. Safe Deletion Logic\n
Execute batch updates to remove the specific variable while preserving the timestamp node.\n

In [10]:
# 🧨 SAFE DELETION EXECUTION
print("🧨 INITIATING BATCH DELETION...")
total_deleted = 0

for station, file_path in backup_files.items():
    df = pd.read_parquet(file_path)
    timestamps = df["timestamp"].tolist()
    
    ref = db.reference(f'auto_weather_stat/{station}/data')
    
    for i in tqdm.tqdm(range(0, len(timestamps), BATCH_SIZE), desc=f"Deleting {station}"):
        batch_ts = timestamps[i:i+BATCH_SIZE]
        payload = {f"{ts}/{TARGET_VARIABLE}": None for ts in batch_ts}
        
        try:
            ref.update(payload)
            total_deleted += len(batch_ts)
        except Exception as e:
            logging.error(f"Error during deletion batch for {station}: {e}")
            print(f"❌ Error during deletion: {e}")

print(f"✅ Successfully deleted {total_deleted} occurrences of {TARGET_VARIABLE}.")
logging.info(f"Deletion complete. Total deleted: {total_deleted}")

🧨 INITIATING BATCH DELETION...
✅ Successfully deleted 0 occurrences of ('temp', 'humi', 'pres').


## 7. Verification Phase\n
Verify that the variable was removed and the parent nodes remain intact.\n

In [23]:
print("🕵️ INITIATING VERIFICATION...")

for station, file_path in backup_files.items():
    print(f"Verifying {station}...")
    df = pd.read_parquet(file_path)
    sample_ts = df["timestamp"].sample(min(5, len(df))).tolist()
    
    for ts in sample_ts:
        record = db.reference(f'auto_weather_stat/{station}/data/{ts}').get()
        if record is None:
            print(f"  ❌ Timestamp {ts} is entirely missing! (CRITICAL ERROR)")
            continue
        
        if TARGET_VARIABLE in record:
            print(f"  ❌ Variable {TARGET_VARIABLE} still exists at {ts}!")
        else:
            other_fields = list(record.keys())
            print(f"  ✅ {ts}: {TARGET_VARIABLE} successfully removed. Intact fields: {other_fields}")

print("Verification complete.")

🕵️ INITIATING VERIFICATION...
Verifying id-05...
  ✅ 1775948008: rain_rate successfully removed. Intact fields: ['dew', 'humidity', 'pressure', 'rainfall', 'rainrate', 'temperature', 'timestamp', 'tips', 'volt']
  ✅ 1776002187: rain_rate successfully removed. Intact fields: ['dew', 'humidity', 'pressure', 'rainfall', 'rainrate', 'temperature', 'timestamp', 'tips', 'volt']
  ✅ 1775966007: rain_rate successfully removed. Intact fields: ['dew', 'humidity', 'pressure', 'rainfall', 'rainrate', 'temperature', 'timestamp', 'tips', 'volt']
  ✅ 1776003267: rain_rate successfully removed. Intact fields: ['dew', 'humidity', 'pressure', 'rainfall', 'rainrate', 'temperature', 'timestamp', 'tips', 'volt']
  ✅ 1775929168: rain_rate successfully removed. Intact fields: ['dew', 'humidity', 'pressure', 'rainfall', 'rainrate', 'temperature', 'timestamp', 'tips', 'volt']
Verification complete.


## 8. Rollback Capability\n
Restore the deleted variables from the Parquet backups if needed.\n

In [17]:
# 🚑 EMERGENCY ROLLBACK
def execute_rollback():
    print("🚑 INITIATING ROLLBACK...")
    total_restored = 0
    
    for station, file_path in backup_files.items():
        if not os.path.exists(file_path):
            print(f"❌ Backup file missing for {station}: {file_path}")
            continue
            
        df = pd.read_parquet(file_path)
        ref = db.reference(f'auto_weather_stat/{station}/data')
        
        for i in tqdm.tqdm(range(0, len(df), BATCH_SIZE), desc=f"Restoring {station}"):
            batch_df = df.iloc[i:i+BATCH_SIZE]
            payload = {f"{row['timestamp']}/{TARGET_VARIABLE}": row[TARGET_VARIABLE] for _, row in batch_df.iterrows()}
            
            try:
                ref.update(payload)
                total_restored += len(batch_df)
            except Exception as e:
                logging.error(f"Error during rollback batch for {station}: {e}")
                print(f"❌ Error during rollback: {e}")
                
    print(f"✅ Rollback complete. Restored {total_restored} occurrences of {TARGET_VARIABLE}.")
    logging.info(f"Rollback complete. Total restored: {total_restored}")

# UNCOMMENT TO EXECUTE ROLLBACK
# execute_rollback()
\n

SyntaxError: unexpected character after line continuation character (231482791.py, line 30)